# FAIR1M ship detection — resolution sweep (Kaggle)

Runs the GPU half of the study: four YOLOv8m detectors on the same 7,706
images and the same 58,982 ship annotations, differing only in image
resolution (100 / 75 / 50 / 25%). Local dataset preparation, annotation
cleaning and preprocessing validation are already complete — this notebook
only trains, evaluates and aggregates, reading the imagery straight from the
attached Kaggle Dataset.

**Before running:**
- Attach the `Fair1m_Ship_Dataset` Kaggle Dataset: Notebook → Add Input →
  Datasets → search "Fair1m_Ship_Dataset".
- Turn on an accelerator: Notebook → Settings → Accelerator → GPU (T4 x2 or
  better).
- Turn Internet ON: Notebook → Settings → Internet. Needed once, to
  `pip install ultralytics` and let it auto-download the COCO-pretrained
  `yolov8m.pt` starting weights.

Then just **Run All**. Steps 0–5 are bootstrap and verification (well under a
minute); step 6 is a cheap GPU sanity run (~1 minute). Step 7 is the training
sweep, and it does **not** try to finish in one run: the full sweep is
12–20 GPU-hours, far more than one Kaggle session or one commit should bet on
completing, so Step 7 does one `TIME_BUDGET_HOURS`-sized slice (default 1.5 h)
and stops itself cleanly — commit (Step 8), then commit this same notebook
again to continue, as many times as it takes. **Every step is safe to
re-run**: finished conditions are skipped, and a condition that was
mid-training resumes from its own checkpoint instead of restarting at epoch 1.
No cell below needs editing.

## Step 0 — Clone the code package

The small validated code package (Python + configs + docs; **not** the
multi-GB imagery, which stays in the attached Kaggle Dataset) lives at
`https://github.com/Ashwin-Prakash-dev/kagg.git` and is cloned below into `/kaggle/working/kaggle`.
Edit the source under `kaggle/` in the project repo, `git push`, and it's
live here on the next run — **no notebook re-upload needed** for a code
change, only for a change to this notebook's own cells (regenerate those with
`scripts/build_kaggle_notebook.py`, don't hand-edit them).

The ~1.5 MB ship-annotation layer (ground-truth boxes, image manifest) is
tracked in that repo for local development, but Step 4 below reconstructs its
own fresh copy directly from the attached dataset's own YOLO labels regardless
— the authoritative source for training is always the attached dataset, never
a possibly-stale committed copy.

In [ ]:
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/Ashwin-Prakash-dev/kagg.git"
PKG = Path("/kaggle/working/kaggle")

if (PKG / ".git").exists():
    print(f"{PKG} already cloned in this session; pulling latest ...")
    subprocess.run(["git", "-C", str(PKG), "pull", "--ff-only"], check=True)
else:
    print(f"Cloning {REPO_URL} -> {PKG} ...")
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(PKG)], check=True)

if str(PKG) not in sys.path:
    sys.path.insert(0, str(PKG))
n_py = sum(1 for _ in PKG.rglob("*.py"))
print(f"\ncode package ready at {PKG}  ({n_py} Python files)")

## Step 1 — Environment

In [ ]:
import os
import platform
import shutil
import sys

import torch

print(f"Python       : {sys.version.split()[0]}")
print(f"Platform     : {platform.platform()}")
print(f"CPU cores    : {os.cpu_count()}  (a sensible default for Step 7's "
      f"WORKERS -- master.yaml's dataloader worker count is pinned higher, "
      f"for a bigger machine)")
print(f"PyTorch      : {torch.__version__}")
try:
    import ultralytics
    print(f"Ultralytics  : {ultralytics.__version__}")
except ImportError:
    print("Ultralytics  : not installed yet (installed in Step 2)")

print(f"CUDA avail.  : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA version : {torch.version.cuda}")
    print(f"GPU count    : {torch.cuda.device_count()}")
    for i in range(torch.cuda.device_count()):
        p = torch.cuda.get_device_properties(i)
        print(f"GPU {i}        : {p.name}  ({p.total_memory / 1e9:.1f} GB)")
else:
    print("No GPU detected. Notebook -> Settings -> Accelerator -> GPU, then "
          "Factory reset & rerun. Training at imgsz=1024 is not viable on CPU.")

free_gb = shutil.disk_usage("/kaggle/working").free / 1e9
print(f"Free disk (/kaggle/working): {free_gb:.1f} GB")

## Step 2 — Install dependencies

Kaggle's GPU image already ships a working `torch` matched to its CUDA
driver; it is deliberately **not** reinstalled here, since replacing it is
the single most common way to break an otherwise-working Kaggle GPU image.
Only `ultralytics` (pinned to the version this study was built against) and a
couple of small, harmless libraries are installed or upgraded.

In [ ]:
!pip install -q "ultralytics==8.4.61" pyyaml pyarrow

import ultralytics
print("ultralytics", ultralytics.__version__)

## Step 3 — Locate the attached Kaggle dataset

`Fair1m_Ship_Dataset` is searched for dynamically under `/kaggle/input` —
this notebook never hard-codes a username or dataset slug, since both are
assigned by Kaggle at upload time. If any of the four resolution conditions
is missing, this stops here rather than failing confusingly deep inside
training.

In [ ]:
from src.paths import CONDITIONS, find_datasets_root

print("=" * 56)
print("FAIR1M SHIP DATASET")
print("=" * 56)
try:
    DATASETS = find_datasets_root()
except FileNotFoundError as e:
    print(f"\nERROR: {e}")
    raise SystemExit(
        "\nSTOPPING: attach 'Fair1m_Ship_Dataset' via Notebook -> Add Input, "
        "then Run All again.")

print(f"\nDataset found under:")
for p in sorted({str(p) for p in DATASETS.values()}):
    print(f"  {p}")
print()
for cond in CONDITIONS:
    label = {"original": "E00", "r75": "E01", "r50": "E02", "r25": "E03"}[cond]
    print(f"{label} ({cond:<8}): FOUND  -> {DATASETS[cond]}")

## Step 4 — Reconstruct the annotation layer, then validate dataset structure

The ~1.5 MB ground-truth annotation layer (image manifest + per-ship boxes in
original-resolution coordinates) isn't embedded in this notebook — see Step 0
— so it's rebuilt here from the attached dataset's own `original`-condition
YOLO labels plus each image's native pixel size. This is lossless: the label
files ARE the source of truth for box geometry (normalized coordinates,
invariant under resize), so reconstructing from them reproduces the same
ground truth the local pipeline validated.

Then confirms every condition has `images/{train,val}` and `labels/{train,val}`,
that `data.yaml` declares exactly one class (`ship`), and — the strongest
check — that the label files are **byte-identical across all four
resolutions**. Normalized YOLO coordinates are invariant under a uniform
resize, so any difference here means the annotation identities are not
shared between conditions and the comparison would be invalid.

In [ ]:
!python {PKG}/scripts/build_annotation_layer.py

In [ ]:
import hashlib

import pandas as pd
import yaml

from src.paths import annotation_dir

images = pd.read_csv(annotation_dir() / "image_manifest.csv")
ships = pd.read_parquet(annotation_dir() / "ship_annotations.parquet")
print(f"annotation layer: {len(images):,} images, {len(ships):,} ship instances\n")

ok = True
ref_digest = None
for cond in ("original", "r75", "r50", "r25"):
    root = DATASETS[cond]
    d = yaml.safe_load((root / "data.yaml").read_text())
    names = list(d["names"].values()) if isinstance(d["names"], dict) else d["names"]
    n_img = len(list((root / "images").rglob("*.jpg")))
    n_lbl = len(list((root / "labels").rglob("*.txt")))
    boxes = sum(sum(1 for l in p.read_text().splitlines() if l.strip())
                for p in (root / "labels").rglob("*.txt"))
    h = hashlib.sha256()
    for p in sorted((root / "labels").rglob("*.txt")):
        h.update(p.read_bytes())
    digest = h.hexdigest()
    if ref_digest is None:
        ref_digest = digest
    same = digest == ref_digest
    class_ok = d.get("nc") == 1 and names == ["ship"]
    good = n_img == len(images) and n_lbl == len(images) and boxes == len(ships) and same and class_ok
    ok &= good
    print(f"  {cond:<9} images {n_img:>6,}  labels {n_lbl:>6,}  boxes {boxes:>7,}  "
          f"nc=1/ship: {class_ok}  labels identical: {same}   {'OK' if good else 'PROBLEM'}")

print("\n" + ("ALL CHECKS PASSED" if ok else "PROBLEMS FOUND -- do not train"))
assert ok

## Step 5 — Paths, resume status, and local pipeline tests

Sets the results root to `/kaggle/working/results` (kept separate from the
`/kaggle/working/kaggle` code package) and looks for a **previously-attached
Notebook Output** of this study under `/kaggle/input` (from an earlier
interrupted session) to restore before continuing — see
`src/paths.restore_previous_results`. `/kaggle/working` alone is **not**
guaranteed to survive a fresh session, which is why this study's own outputs
double as a re-attachable input.

Then runs the local self-test suite (`scripts/verify_kaggle_setup.py`) —
structural and logic checks that need neither a GPU nor the dataset, plus the
dataset checks now that Steps 3/4 confirmed the attachment.

In [ ]:
from src.paths import find_results_root, restore_previous_results

RESULTS_ROOT = find_results_root()
restored_from = restore_previous_results(RESULTS_ROOT)
if restored_from:
    print(f"Restored previous results from {restored_from} -> {RESULTS_ROOT}")
else:
    print(f"No previous results found to restore; starting fresh at {RESULTS_ROOT}")

print()
run_ids = [("E00", "00_baseline"), ("E01", "01_r75"), ("E02", "02_r50"), ("E03", "03_r25")]
for label, run_id in run_ids:
    out_dir = RESULTS_ROOT / run_id
    if (out_dir / "metrics.json").exists():
        state = "COMPLETE"
    elif (out_dir / "train" / run_id / "weights" / "last.pt").exists():
        state = "RESUME (checkpoint found)"
    else:
        state = "NOT STARTED"
    print(f"  {label} {run_id:<12} {state}")

In [ ]:
!python {PKG}/scripts/verify_kaggle_setup.py

## Step 6 — GPU sanity test (required before the full sweep)

One epoch on a small, curated, ship-guaranteed subset (32 train / 16 val
images per condition), at the study's real `imgsz=1024` / `max_det=1000`.
Proves the plumbing on this GPU before committing hours to it: CUDA works,
the model initializes, images and labels load, training starts, loss is
produced, validation runs, predictions are generated, a checkpoint is saved,
and metrics come out finite.

This is a small staged subset rather than a `--fraction` of the full
condition on purpose: Ultralytics scans and verifies every image/label pair
in `train:`/`val:` to build its label cache *before* `fraction` is ever
applied, and that cache can't be saved and reused on the read-only
`/kaggle/input` mount -- so a "5%" run pointed at the full ~7,706-image
condition would still pay the full scan cost, on every launch, across all
four conditions, which is what made this step slow. See
`src/paths.stage_sanity_subset`.

**These numbers are not results.** They go to `*_sanity` directories and are
excluded from the aggregated tables. If the assertion below fails, **stop**
— do not proceed to Step 7.

Runs once: if a previous commit's restored Output already has passing sanity
results, this is skipped (no `--force`) rather than repeated every commit.

In [ ]:
!python {PKG}/scripts/run_all.py --sanity --epochs 1 --results-root {RESULTS_ROOT}

In [ ]:
import json

ok = True
for _, run_id in run_ids:
    p = RESULTS_ROOT / f"{run_id}_sanity" / "metrics.json"
    if not p.exists():
        print(f"{run_id}_sanity: NO metrics.json -- FAIL"); ok = False
        continue
    m = json.loads(p.read_text()).get("metrics", {})
    n_gt = m.get("n_gt", {}).get("all", 0)
    has_ckpt = (RESULTS_ROOT / f"{run_id}_sanity" / "best.pt").exists()
    good = n_gt > 0 and has_ckpt
    print(f"{run_id}_sanity   mAP50 {m.get('mAP50', float('nan')):.4f}   "
          f"detections {m.get('n_detections', 0):,}   n_gt {n_gt:,}   "
          f"checkpoint saved: {has_ckpt}")
    ok &= good

print("\n" + ("SANITY TEST PASSED -- safe to run the full sweep (Step 7)"
              if ok else "SANITY TEST FAILED -- investigate before proceeding"))
assert ok, "Do not begin the full sweep: the sanity test did not pass."

## Step 7 — Time-budgeted training slice: E00 → E01 → E02 → E03

Uses the **full** 7,706-image, 58,982-ship dataset (not the sanity subset).
The full sweep is roughly 3–5 GPU-hours per condition on a T4 (12–20 h total)
— far more than fits in one Kaggle session or one commit you'd want to bet on
completing. This is the expected cost of the design (`imgsz=1024`, 50 epochs,
early stopping deliberately disabled — see `configs/master.yaml`), not a sign
anything is wrong.

**This cell is meant to be committed (Save Version → Save & Run All)
repeatedly**, not run once. Each run does up to `TIME_BUDGET_HOURS` of work —
resuming whatever condition is in progress, then starting the next one with
whatever time is left — and stops itself cleanly before the budget runs out,
rather than risking Kaggle killing it mid-epoch. Pick a budget comfortably
under your session limit (1–1.5 h leaves plenty of margin for Steps 0–6 and
any slow epoch). Re-commit this same notebook until the status cell below
reports all four conditions complete.

`BATCH` and `WORKERS` are **environment** knobs, not experimental ones — see
`src/training.ALLOWED_OVERRIDES` / `ENV_KEYS`, which mechanically forbid
overriding anything that would actually change the comparison (`imgsz`,
`epochs`, augmentation, LR, etc. cannot be touched this way). Both are fixed
once and used identically for all four conditions. Two safe ways to speed
individual runs up if they're GPU- or CPU-bound:

- **`BATCH`** — raise it if `nvidia-smi` (Step 1) shows headroom below the
  GPU's VRAM (T4: 15 GB; YOLOv8m at imgsz=1024/batch=8 typically uses well
  under that). A larger batch improves GPU utilization. If it OOMs, Ultralytics
  errors clearly — just lower it back down and rerun (already-finished
  conditions are skipped, so nothing is lost).
- **`WORKERS`** — `configs/master.yaml` pins 8 dataloader workers, tuned for
  a many-core machine. Kaggle GPU sessions typically have far fewer CPU cores;
  oversubscribing workers can add contention overhead rather than help. Try
  matching it to the actual core count (`!nproc` or the Step 1 output).

In [ ]:
BATCH = 8              # keep identical for all four conditions; see configs/master.yaml
WORKERS = 4            # match this to the CPU cores Kaggle actually gave this session
TIME_BUDGET_HOURS = 1.5   # this run stops cleanly at or before this many hours

!python {PKG}/scripts/run_all.py --results-root {RESULTS_ROOT} \
    --batch {BATCH} --workers {WORKERS} --time-budget {TIME_BUDGET_HOURS}

In [ ]:
import json

status_path = RESULTS_ROOT / "experiment_status.json"
status = json.loads(status_path.read_text()) if status_path.exists() else {}
done = [k for k, v in status.items() if v["status"] == "ok" or v["status"].startswith("skipped")]

print("=" * 56)
print("PROGRESS")
print("=" * 56)
for run_id in [r for _, r in run_ids]:
    s = status.get(run_id, {}).get("status", "not started")
    print(f"  {run_id:<14} {s}")

if len(done) == len(run_ids):
    print("\nALL FOUR CONDITIONS COMPLETE -- proceed to Step 8.")
else:
    print(f"\n{len(done)}/{len(run_ids)} complete. NOT finished yet.")
    print("Proceed to Step 8 to persist this progress, then come back and "
          "commit this notebook again (Step 7 onward) to continue -- it will "
          "resume exactly where this run stopped.")

## Step 8 — Persist progress as a Kaggle Notebook Output

`/kaggle/working` is not guaranteed to survive a fresh session. After every
Step 7 run (finished or not), use Kaggle's **Save Version → Save & Run All
(Commit)** to persist everything currently in `/kaggle/working` as this
notebook's Output. This is the mechanism this whole workflow depends on:
Kaggle's "Save Version" always re-executes the notebook fresh in a new
container rather than snapshotting a live session, so a run's progress is
IN THE OUTPUT SAVED HERE or it doesn't exist for next time.

To resume, whether in a brand-new session or the next commit: **Add Input →
Notebook Output → this notebook's latest version** (in addition to
`Fair1m_Ship_Dataset`), enable GPU, and Run All — Step 5 finds and restores
that output automatically, and Step 7 picks up exactly where it left off
(skips finished conditions, resumes the in-progress one, starts the next).

**The loop, in short:** commit → check the Step 7 status cell → if not all
four are done, reattach this notebook's own latest Output and commit again →
repeat until done → Step 9 onward.

In [ ]:
print("Currently in", RESULTS_ROOT, ":")
for p in sorted(RESULTS_ROOT.rglob("*")):
    if p.is_file():
        print(" ", p.relative_to(RESULTS_ROOT))

## Step 9 — Collect results (tables + plots)

Builds the resolution × object-size table, degradation percentages, and
plots. A condition that has not run is reported as **missing**, never filled
in or estimated.

In [ ]:
!python {PKG}/scripts/collect_results.py --results-root {RESULTS_ROOT} --out {RESULTS_ROOT}

In [ ]:
import pandas as pd
from IPython.display import Image, display

df = pd.read_csv(RESULTS_ROOT / "master_results.csv")
cols = ["resolution", "mAP50", "mAP50_95", "AP_small", "AP_medium", "AP_large",
        "recall", "precision"]
print("PERFORMANCE (size bins fixed at ORIGINAL resolution:")
print("  small < 1024 px^2, medium 1024-9216, large >= 9216)\n")
display(df[[c for c in cols if c in df]].round(4))

drop = [c for c in df.columns if c.endswith("_drop_pct")]
if drop:
    print("\nDEGRADATION vs baseline (%):")
    display(df[["resolution"] + drop].round(2))

for name in ("resolution_x_object_size.png", "resolution_vs_map.png",
             "training_cost.png"):
    p = RESULTS_ROOT / "plots" / name
    if p.exists():
        display(Image(filename=str(p)))

## Step 10 — Qualitative comparisons

Renders the same validation scenes (small ships, medium ships, large ships,
dense harbours, isolated ships, difficult backgrounds) across all four
resolutions with each condition's own model's predictions overlaid on the
shared ground truth.

In [ ]:
!python {PKG}/scripts/qualitative_comparison.py --results-root {RESULTS_ROOT} \
    --out {RESULTS_ROOT}/qualitative

## Step 11 — Experiment report

In [ ]:
!python {PKG}/scripts/build_experiment_report.py --results-root {RESULTS_ROOT}

## Step 12 — Package a compact results archive

One zip with metrics, configs, logs, plots, qualitative results, the report,
and the best/last checkpoints for each condition. **Does not include the
image datasets** — those already live in the attached Kaggle Dataset, so
duplicating 6+ GB here would be pure waste.

In [ ]:
import zipfile
from pathlib import Path

archive_path = Path("/kaggle/working/fair1m_ship_resolution_results.zip")
with zipfile.ZipFile(archive_path, "w", zipfile.ZIP_DEFLATED) as zf:
    for p in RESULTS_ROOT.rglob("*"):
        if p.is_file():
            zf.write(p, arcname=str(p.relative_to(RESULTS_ROOT.parent)))
    for cfg in (PKG / "configs").glob("*.yaml"):
        zf.write(cfg, arcname=f"configs/{cfg.name}")

size_mb = archive_path.stat().st_size / 1e6
print(f"Wrote {archive_path}  ({size_mb:.1f} MB)")
print("This archive, plus the Notebook Output from Step 8, is everything "
      "needed to preserve the experimental results without duplicating the "
      "attached dataset.")

## Optional follow-ons

Only after the primary sweep is complete and its results are recorded.

- **Repeated seeds.** One run per condition supports no significance claim.
  If GPU budget allows, repeat E00/E02/E03 with seeds 123 and 456 and report
  mean ± sd. If not, state plainly that the study is single-seed.
- **Cross-resolution transfer.** `scripts/evaluate.py --eval-on <cond>
  --tag ...` scores a model trained at one resolution on another. A different
  question from the primary sweep; keep it separately tagged so
  `collect_results.py` cannot mistake it for a primary-sweep number.

In [ ]:
# Repeated seeds -- uncomment to run. Costs roughly 2x the primary sweep.
# for seed in (123, 456):
#     for cfg in ("baseline", "r50", "r25"):
#         !python {PKG}/scripts/run_experiment.py --config {PKG}/configs/{cfg}.yaml \
#             --results-root {RESULTS_ROOT}_seed{seed} --batch {BATCH}